# CUDA Concept and Programming
## Group 3:

Members:
*   Alcasid, Aizy
*   Bilanes, Kendrick
*   Corpuz, Gerald
*   Roa, Luis







CUDA Version 1: C/C++ Program

In [ ]:
%%writefile C_output.c

// allows usage of real time extension features
#define _POSIX_C_SOURCE 199309L

#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <time.h>

#define NUM_RUNS 30 // Minimum 30 runs

static FILE* g_csv = NULL;   // per-run times (n,run,time_ms) for plotting

// C Kernel
void axpy_c(int n, float a, const float* X, const float* Y, float* Z) {
    int i;
    for (i = 0; i < n; i++) {
        Z[i] = a * X[i] + Y[i];
    }
}

// HELPER FUNCTIONS:

// Prints elements (First 5 and Last 5)
void print_samples(const char* kernel_name, const float Z[], int n) {
    printf("%s Elements:\n  First 5: %.4f %.4f %.4f %.4f %.4f\n",
        kernel_name, Z[0], Z[1], Z[2], Z[3], Z[4]);
    printf("  Last 5:  %.4f %.4f %.4f %.4f %.4f\n",
        Z[n - 5], Z[n - 4], Z[n - 3], Z[n - 2], Z[n - 1]);
}

// Testbench
void run_test(int n, float a) {
    printf("\n====================================================================\n");
    printf(" Testing Vector Size n = %d (Scalar a = %.2f)\n", n, a);
    printf("====================================================================\n");

    int i, run;
    size_t bytes = (size_t)n * sizeof(float);

    float* X = (float*)malloc(bytes);
    float* Y = (float*)malloc(bytes);
    float* Z_c = (float*)malloc(bytes);

    if (!X || !Y || !Z_c) {
        printf("Memory allocation failed!\n");
        free(X); free(Y); free(Z_c);
        return;
    }

    // Initialize vectors
    for (i = 0; i < n; i++) {
        X[i] = sinf(i * 0.001f);
        Y[i] = cosf(i * 0.002f);
        Z_c[i] = 0.0f;
    }

    // C Kernel Execution (NUM_RUNS times) using Linux clock_gettime
    double times[NUM_RUNS];
    for (run = 0; run < NUM_RUNS; run++) {
        struct timespec start, end;
        clock_gettime(CLOCK_MONOTONIC, &start);
        axpy_c(n, a, X, Y, Z_c);
        clock_gettime(CLOCK_MONOTONIC, &end);

        times[run] = ((end.tv_sec - start.tv_sec) * 1000.0) + ((end.tv_nsec - start.tv_nsec) / 1000000.0);
        if (g_csv) fprintf(g_csv, "%d,%d,%.6f\n", n, run + 1, times[run]);
    }

    // Statistics
    double sum_all = 0.0, sum_warm = 0.0;
    double tmin = times[0], tmax = times[0];
    for (run = 0; run < NUM_RUNS; run++) {
        sum_all += times[run];
        if (run > 0) sum_warm += times[run];
        if (times[run] < tmin) tmin = times[run];
        if (times[run] > tmax) tmax = times[run];
    }
    double avg_all = sum_all / NUM_RUNS;
    double avg_warm = sum_warm / (NUM_RUNS - 1);

    printf("\nC Version:\n");
    printf("Time = %.6f ms   (average of all %d runs)\n", avg_all, NUM_RUNS);
    printf("Run 1                    = %.6f ms\n", times[0]);
    printf("Runs 2-%d avg            = %.6f ms\n", NUM_RUNS, avg_warm);
    printf("Min / Max                = %.6f / %.6f ms\n", tmin, tmax);
    print_samples("C Vector Z", Z_c, n);

    // Cleans the vectors
    free(X);
    free(Y);
    free(Z_c);
}

// Main
int main(int argc, char* argv[]) {
    float a = 2.0f;

    printf("Data init: X[i] = sinf(i*0.001f), Y[i] = cosf(i*0.002f), Z[i] = 0, a = %.1f\n", a);

    g_csv = fopen("c_runs.csv", "w");
    if (g_csv) fprintf(g_csv, "n,run,time_ms\n");

    if (argc > 1) {
        // Single size from the command line
        int n = atoi(argv[1]);
        if (n < 5) {
            printf("n must be at least 5.\n");
            return 1;
        }
        run_test(n, a);
    }
    else {
        // Vector Sizes: 2^20, 2^26, 2^28 + boundary values
        int test_sizes[] = {
            1 << 20,       // 2^20 = 1,048,576
            1 << 24,       // 2^24 = 16,777,216
            1 << 26,       // 2^26 = 67,108,864
            1 << 28,       // 2^28 = 268,435,456
            1003,          // Boundary
            (1 << 20) + 7  // Boundary
        };

        int num_tests = sizeof(test_sizes) / sizeof(test_sizes[0]);

        for (int t = 0; t < num_tests; t++) {
            run_test(test_sizes[t], a);
        }
    }

    if (g_csv) fclose(g_csv);
    return 0;
}

Overwriting C_output.c


In [ ]:
!gcc -O2 -fopenmp C_output.c -o C_output -lm
!./C_output

Data init: X[i] = sinf(i*0.001f), Y[i] = cosf(i*0.002f), Z[i] = 0, a = 2.0

 Testing Vector Size n = 1048576 (Scalar a = 2.00)

C Version:
Time = 1.683082 ms   (average of all 30 runs)
Run 1                    = 1.750474 ms
Runs 2-30 avg            = 1.680758 ms
Min / Max                = 1.432016 / 1.888755 ms
C Vector Z Elements:
  First 5: 1.0000 1.0020 1.0040 1.0060 1.0080
  Last 5:  -1.1915 -1.1881 -1.1847 -1.1808 -1.1774

 Testing Vector Size n = 16777216 (Scalar a = 2.00)

C Version:
Time = 23.780529 ms   (average of all 30 runs)
Run 1                    = 27.605954 ms
Runs 2-30 avg            = 23.648618 ms
Min / Max                = 19.297711 / 28.231964 ms
C Vector Z Elements:
  First 5: 1.0000 1.0020 1.0040 1.0060 1.0080
  Last 5:  1.1895 1.1882 1.1882 1.1868 1.1868

 Testing Vector Size n = 67108864 (Scalar a = 2.00)

C Version:
Time = 79.495203 ms   (average of all 30 runs)
Run 1                    = 82.467531 ms
Runs 2-30 avg            = 79.392709 ms
Min / Max           